# Reset Bronze Layer

Truncates all bronze Delta tables so you can re-run a full load from `01_landing_to_bronze` without accumulating duplicates.

**Usage:** Run this notebook before each full-load re-run. This does NOT drop the tables — it keeps the schema and empties the rows.

In [0]:
%run ./00_config_and_logging

In [0]:
# Tables to reset (bronze layer + optional log table)
bronze_tables = [
    ("bronze_nvd",    BRONZE_NVD),
    ("bronze_epss",   BRONZE_EPSS),
    ("bronze_kev",    BRONZE_KEV),
    ("bronze_assets", BRONZE_ASSETS),
]

reset_log = True  # set to False to keep pipeline execution logs

if reset_log:
    bronze_tables.append(("pipeline_execution_logs", LOG_TABLE))

print(f"Will reset {len(bronze_tables)} table(s):")
for name, tbl in bronze_tables:
    print(f"  - {name}  ->  {tbl}")

In [0]:
from pyspark.sql.functions import count as count_rows

for name, tbl in bronze_tables:
    try:
        # Check if table exists
        spark.sql(f"DESCRIBE TABLE {tbl}")
        # Count rows before truncation
        before = spark.table(tbl).count()
        # Truncate (keeps schema, removes all rows)
        spark.sql(f"TRUNCATE TABLE {tbl}")
        print(f"✓ {name}: truncated {before} rows")
    except Exception as e:
        print(f"⚠ {name}: skipped ({str(e)[:80]})")

In [0]:
print("=== Verification ===")
for name, tbl in bronze_tables:
    try:
        cnt = spark.table(tbl).count()
        print(f"{name}: {cnt} rows")
    except Exception:
        print(f"{name}: table does not exist yet")